# Data Ingestion

In [1]:
import os 
%pwd

'd:\\Data Science\\Codes\\02 MLOPs\\08 MLOPs Project Heart Disease Prediction\\research'

In [2]:
os.chdir("../")
%pwd

'd:\\Data Science\\Codes\\02 MLOPs\\08 MLOPs Project Heart Disease Prediction'

In [3]:
from dataclasses import dataclass 
from pathlib import Path 

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir : Path 
    source_URL : str
    local_data_file : Path
    unzip_dir  : Path 

In [ ]:
from mlProject.constants import * 
from mlProject.utils.common import read_yaml,create_directories

In [5]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )

        return data_ingestion_config

In [ ]:
from mlProject.utils.common import get_size
from mlProject import logger 
from urllib import request  
import zipfile
import os 

In [18]:
class DataIngestion:
    def __init__(self,config: DataIngestionConfig):
        self.config=config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename,headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exist of size: {get_size(Path(self.config.local_data_file))}")
    
    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path,exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file,"r") as zip_ref:
            zip_ref.extractall(unzip_path)

In [19]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e 

[2026-03-03 19:35:43,425: INFO: common: YAML file loaded successfully: config\config.yaml:]
[2026-03-03 19:35:43,432: INFO: common: YAML file loaded successfully: params.yaml:]
[2026-03-03 19:35:43,432: INFO: common: YAML file loaded successfully: schema.yaml:]
[2026-03-03 19:35:43,438: INFO: common: Directory created at: artifacts:]
[2026-03-03 19:35:43,438: INFO: common: Directory created at: artifacts/data_ingestion:]
[2026-03-03 19:35:43,438: INFO: 3076577460: File already exist of size: ~ 6695 KB:]
